# Sentiment-Based Sorting — Merged Library Domain Notebook

Naive Bayes, Decision Tree, and a Naive Bayes -> Decision Tree hybrid.
Metrics: accuracy, F1-score, recall.

**Dataset:** `merged_library_dataset.csv` — 7,124 rows combining EduRABSA
(6,491 rows, general education reviews) with local six-category student
feedback (633 rows, including library facilities).

Columns: `text`, `sentiment` (positive/neutral/negative), `source`
(edurabsa or local), `category` (edurabsa, or one of the six local
categories including library_facilities).

Section 13 evaluates on the library-facilities rows specifically, since
that is the deployment target, not the merged average. Section 14 adds
back-translation augmentation to the library-facilities training rows
only, and re-evaluates.


## 1. Setup and load


In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report, confusion_matrix
import joblib

RANDOM_STATE = 42

df = pd.read_csv('merged_library_dataset.csv')

print('Shape:', df.shape)
print()
print(df['source'].value_counts())
print()
print(df['category'].value_counts())
print()
print(df['sentiment'].value_counts())
df.head()


ModuleNotFoundError: No module named 'pandas'

## 2. Clean the text


In [ ]:
df = df.drop_duplicates(subset='text').reset_index(drop=True)

def preprocess(text):
    text = re.sub(r'[^a-zA-Z\s]', ' ', str(text).lower())
    return ' '.join(text.split())

df['clean'] = df['text'].apply(preprocess)
df = df[df['clean'].str.len() > 0].reset_index(drop=True)
print('Rows after cleaning:', len(df))
print(df['source'].value_counts())


## 3. Split into train and test

80/20, stratified by sentiment across the merged dataset.


In [ ]:
train, test = train_test_split(
    df, test_size=0.20, stratify=df['sentiment'], random_state=RANDOM_STATE)

X_train, y_train = train['clean'], train['sentiment']
X_test,  y_test  = test['clean'],  test['sentiment']

vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=2)
Xtr = vectorizer.fit_transform(X_train)
Xte = vectorizer.transform(X_test)

print('Train size:', Xtr.shape[0], '  Test size:', Xte.shape[0])
print('Test class counts:', y_test.value_counts().to_dict())
print('Test rows by source:', test['source'].value_counts().to_dict())
print('Library-facilities rows in test:', (test['category'] == 'library_facilities').sum())
print('Library-facilities rows in train:', (train['category'] == 'library_facilities').sum())


## 4. Naive Bayes


In [ ]:
nb = MultinomialNB(alpha=0.3)
nb.fit(Xtr, y_train)
nb_pred = nb.predict(Xte)

print('Accuracy :', round(accuracy_score(y_test, nb_pred), 4))
print('F1-score :', round(f1_score(y_test, nb_pred, average='macro'), 4))
print('Recall   :', round(recall_score(y_test, nb_pred, average='macro'), 4))
print()
print(classification_report(y_test, nb_pred, digits=3))


## 5. Decision Tree


In [ ]:
dt = DecisionTreeClassifier(class_weight='balanced', min_samples_leaf=2, random_state=RANDOM_STATE)
dt.fit(Xtr, y_train)
dt_pred = dt.predict(Xte)

print('Accuracy :', round(accuracy_score(y_test, dt_pred), 4))
print('F1-score :', round(f1_score(y_test, dt_pred, average='macro'), 4))
print('Recall   :', round(recall_score(y_test, dt_pred, average='macro'), 4))
print()
print(classification_report(y_test, dt_pred, digits=3))


## 6. Hybrid (Naive Bayes -> Decision Tree)


In [ ]:
class Hybrid(BaseEstimator, ClassifierMixin):
    def __init__(self, threshold=0.9):
        self.threshold = threshold

    def fit(self, X, y):
        self.nb_ = MultinomialNB(alpha=0.3).fit(X, y)
        self.dt_ = DecisionTreeClassifier(class_weight='balanced', min_samples_leaf=2,
                                          random_state=RANDOM_STATE).fit(X, y)
        return self

    def predict(self, X):
        proba = self.nb_.predict_proba(X)
        pred = self.nb_.predict(X).copy()
        low_confidence = proba.max(axis=1) < self.threshold
        if low_confidence.any():
            pred[low_confidence] = self.dt_.predict(X[low_confidence])
        return pred

hybrid = Hybrid(threshold=0.9)
hybrid.fit(Xtr, y_train)
hybrid_pred = hybrid.predict(Xte)

print('Accuracy :', round(accuracy_score(y_test, hybrid_pred), 4))
print('F1-score :', round(f1_score(y_test, hybrid_pred, average='macro'), 4))
print('Recall   :', round(recall_score(y_test, hybrid_pred, average='macro'), 4))
print()
print(classification_report(y_test, hybrid_pred, digits=3))


## 7. Compare all three


In [ ]:
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'Decision Tree', 'Hybrid'],
    'Accuracy': [accuracy_score(y_test, nb_pred), accuracy_score(y_test, dt_pred), accuracy_score(y_test, hybrid_pred)],
    'F1-score': [f1_score(y_test, nb_pred, average='macro'), f1_score(y_test, dt_pred, average='macro'), f1_score(y_test, hybrid_pred, average='macro')],
    'Recall':   [recall_score(y_test, nb_pred, average='macro'), recall_score(y_test, dt_pred, average='macro'), recall_score(y_test, hybrid_pred, average='macro')],
}).set_index('Model').round(4)
results


In [ ]:
results.plot(kind='bar', figsize=(8, 5), rot=0)
plt.title('Model comparison (merged dataset)')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 8. Effect of text preprocessing (Objective 1)


In [ ]:
vectorizer_raw = CountVectorizer(ngram_range=(1, 2), min_df=2)
Xtr_raw = vectorizer_raw.fit_transform(train['text'])
Xte_raw = vectorizer_raw.transform(test['text'])

nb_raw = MultinomialNB(alpha=0.3).fit(Xtr_raw, y_train)
dt_raw = DecisionTreeClassifier(class_weight='balanced', min_samples_leaf=2, random_state=RANDOM_STATE).fit(Xtr_raw, y_train)
hybrid_raw = Hybrid(threshold=0.9).fit(Xtr_raw, y_train)

nb_raw_pred = nb_raw.predict(Xte_raw)
dt_raw_pred = dt_raw.predict(Xte_raw)
hybrid_raw_pred = hybrid_raw.predict(Xte_raw)

preprocessing_effect = pd.DataFrame([
    {'Preprocessing': 'Raw (uncleaned)', 'Model': 'Naive Bayes', 'Accuracy': accuracy_score(y_test, nb_raw_pred), 'F1-score': f1_score(y_test, nb_raw_pred, average='macro')},
    {'Preprocessing': 'Raw (uncleaned)', 'Model': 'Decision Tree', 'Accuracy': accuracy_score(y_test, dt_raw_pred), 'F1-score': f1_score(y_test, dt_raw_pred, average='macro')},
    {'Preprocessing': 'Raw (uncleaned)', 'Model': 'Hybrid', 'Accuracy': accuracy_score(y_test, hybrid_raw_pred), 'F1-score': f1_score(y_test, hybrid_raw_pred, average='macro')},
    {'Preprocessing': 'Cleaned', 'Model': 'Naive Bayes', 'Accuracy': accuracy_score(y_test, nb_pred), 'F1-score': f1_score(y_test, nb_pred, average='macro')},
    {'Preprocessing': 'Cleaned', 'Model': 'Decision Tree', 'Accuracy': accuracy_score(y_test, dt_pred), 'F1-score': f1_score(y_test, dt_pred, average='macro')},
    {'Preprocessing': 'Cleaned', 'Model': 'Hybrid', 'Accuracy': accuracy_score(y_test, hybrid_pred), 'F1-score': f1_score(y_test, hybrid_pred, average='macro')},
]).round(4)
preprocessing_effect


In [ ]:
delta = preprocessing_effect.pivot(index='Model', columns='Preprocessing', values='Accuracy')
delta['Accuracy change'] = delta['Cleaned'] - delta['Raw (uncleaned)']
print(delta.round(4))


## 9. Save the model for Django


In [ ]:
joblib.dump(hybrid, 'sentiment_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print('Saved: sentiment_model.pkl, vectorizer.pkl')

loaded_model = joblib.load('sentiment_model.pkl')
loaded_vec   = joblib.load('vectorizer.pkl')

for t in ['the aircon on the 2nd floor is too warm', 'the librarian helped me find the book right away']:
    cleaned = preprocess(t)
    pred = loaded_model.predict(loaded_vec.transform([cleaned]))[0]
    print(f'{pred:9} <- {t}')


## 10. Confusion matrices


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels = ['negative', 'neutral', 'positive']
for ax, name, pred in zip(axes, ['Naive Bayes', 'Decision Tree', 'Hybrid'], [nb_pred, dt_pred, hybrid_pred]):
    cm = confusion_matrix(y_test, pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False, xticklabels=labels, yticklabels=labels)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()


## 11. Cross-validation on the merged training set


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in [('Naive Bayes', MultinomialNB(alpha=0.3)),
                    ('Decision Tree', DecisionTreeClassifier(class_weight='balanced', min_samples_leaf=2, random_state=RANDOM_STATE)),
                    ('Hybrid', Hybrid(threshold=0.9))]:
    pipe = Pipeline([('v', CountVectorizer(ngram_range=(1,2), min_df=2)), ('c', model)])
    acc = cross_val_score(pipe, train['clean'], train['sentiment'], cv=cv, scoring='accuracy')
    f1m = cross_val_score(pipe, train['clean'], train['sentiment'], cv=cv, scoring='f1_macro')
    cv_rows.append({'Model': name, 'CV Accuracy': f'{acc.mean():.4f} +/- {acc.std():.4f}', 'CV Macro F1': f'{f1m.mean():.4f} +/- {f1m.std():.4f}'})
pd.DataFrame(cv_rows).set_index('Model')


## 12. Sort feedback by sentiment priority


In [ ]:
SENTIMENT_PRIORITY = {'negative': 0, 'neutral': 1, 'positive': 2}

def sort_feedback(texts, model=None, vec=None):
    model = model or hybrid
    vec = vec or vectorizer
    cleaned = [preprocess(t) for t in texts]
    V = vec.transform(cleaned)
    labels_ = model.predict(V)
    out = pd.DataFrame({'feedback': texts, 'sentiment': labels_})
    out['priority'] = out['sentiment'].map(SENTIMENT_PRIORITY)
    return out.sort_values('priority').drop(columns='priority').reset_index(drop=True)

sample = [
    "the wifi keeps dropping on the 2nd floor quiet study area",
    "the librarian at the circulation desk was very helpful",
    "the collection has enough books but the database access is limited",
    "i never got a reply about the reserved book i requested",
    "borrowing and returning was fast and easy, thank you",
]
sort_feedback(sample)


## 13. Library-facilities performance

The actual deployment target. Isolates the library-facilities rows in the
test set and reports each model's performance on them specifically.


In [ ]:
lib_mask = test['category'] == 'library_facilities'
lib_test = test[lib_mask]
lib_Xte = vectorizer.transform(lib_test['clean'])
lib_ytrue = lib_test['sentiment']

print('Library-facilities rows in test set:', len(lib_test))
print(lib_ytrue.value_counts().to_dict())
print()

for name, model in [('Naive Bayes', nb), ('Decision Tree', dt), ('Hybrid', hybrid)]:
    pred = model.predict(lib_Xte)
    acc = accuracy_score(lib_ytrue, pred)
    f1 = f1_score(lib_ytrue, pred, average='macro', zero_division=0)
    print(f'{name:15} accuracy={acc:.4f}  macro F1={f1:.4f}')

print()
print('--- Hybrid detail on library-facilities rows only ---')
lib_pred = hybrid.predict(lib_Xte)
print(classification_report(lib_ytrue, lib_pred, digits=3, zero_division=0))


## 14. Back-translation augmentation for library-facilities feedback

**What this does.** Each library-facilities comment in the *training* set
is translated English -> Tagalog -> English. The round trip usually
reworks the sentence into slightly different wording while keeping the
same meaning, giving you a paraphrased duplicate with the same label.
This is applied to training rows only, and only to the library-facilities
category — it is meant to help the class with the fewest examples in your
actual deployment domain, not to inflate the dataset generally.

**What this does not do.** It does not add real new information — the
model still learns from the same underlying library comments, just
phrased two ways each. It does not fix domain shift; it only reworks
text you already have. And for very short comments (a few of your
library rows are one or two words), the round trip can come back
nearly identical, or in rare cases distort the meaning — negation is
the classic failure point in machine translation ("not good" round-
tripping into something that reads more neutral or positive). **Spot-
check a sample of the augmented rows before trusting the results.**

**Why this section cannot be verified from this notebook-writing
session.** Back-translation needs a live call to a translation service.
That requires outbound internet access to Google's or another
provider's translation servers. This step will run in Google Colab or
on your own machine with normal internet access; it will not run in a
fully offline or restricted environment. Run this section, read the
printed before/after comparison, and check a handful of the augmented
rows yourself before citing the result.


In [ ]:
# Run once.
# !pip install deep-translator


SyntaxError: invalid syntax (881278091.py, line 2)

In [ ]:
from deep_translator import GoogleTranslator
import time

def back_translate(text, pivot_lang='tl', pause=0.3):
    """English -> pivot language -> English. Returns None on failure
    instead of raising, so one bad request does not stop the whole loop."""
    try:
        to_pivot = GoogleTranslator(source='en', target=pivot_lang).translate(text)
        if not to_pivot:
            return None
        back_to_en = GoogleTranslator(source=pivot_lang, target='en').translate(to_pivot)
        time.sleep(pause)  # be polite to the free endpoint - avoid rate limiting
        return back_to_en
    except Exception as e:
        print(f'  [skipped] {text[:50]!r} -> {type(e).__name__}')
        return None


In [ ]:
lib_train = train[train['category'] == 'library_facilities'].copy()
print(f'Back-translating {len(lib_train)} library-facilities training rows...')
print('This calls an external translation service per row - expect roughly')
print('1-2 rows per second with the pause above, so a few minutes total.\n')

augmented_rows = []
for _, row in lib_train.iterrows():
    bt_text = back_translate(row['text'])
    if bt_text and bt_text.strip().lower() != row['text'].strip().lower():
        augmented_rows.append({
            'text': bt_text, 'sentiment': row['sentiment'],
            'source': 'local_backtranslated', 'category': 'library_facilities'
        })

augmented_df = pd.DataFrame(augmented_rows)
print(f'\nProduced {len(augmented_df)} usable augmented rows '
      f'out of {len(lib_train)} originals '
      f'({len(lib_train) - len(augmented_df)} skipped or unchanged).')
augmented_df.head(10)


In [ ]:
# Spot-check a handful side by side before trusting any of this.
check = lib_train.reset_index(drop=True).iloc[:len(augmented_df)].reset_index(drop=True)
compare = pd.DataFrame({
    'original': check['text'].values[:len(augmented_df)],
    'back_translated': augmented_df['text'].values,
    'label': check['sentiment'].values[:len(augmented_df)],
})
compare.head(15)


### Retrain with the augmented rows added

Same preprocessing, same vectorizer settings, same test set — only the
training data changes: the original training set plus the back-translated
library-facilities rows.


In [ ]:
augmented_df['clean'] = augmented_df['text'].apply(preprocess)
augmented_df = augmented_df[augmented_df['clean'].str.len() > 0]

train_bt = pd.concat([train, augmented_df[['text', 'sentiment', 'source', 'category', 'clean']]],
                     ignore_index=True)

print('Original training rows :', len(train))
print('Augmented training rows:', len(train_bt))

vectorizer_bt = CountVectorizer(ngram_range=(1, 2), min_df=2)
Xtr_bt = vectorizer_bt.fit_transform(train_bt['clean'])
Xte_bt = vectorizer_bt.transform(test['clean'])   # same untouched test set

nb_bt = MultinomialNB(alpha=0.3).fit(Xtr_bt, train_bt['sentiment'])
dt_bt = DecisionTreeClassifier(class_weight='balanced', min_samples_leaf=2, random_state=RANDOM_STATE).fit(Xtr_bt, train_bt['sentiment'])
hybrid_bt = Hybrid(threshold=0.9).fit(Xtr_bt, train_bt['sentiment'])


### Compare: with vs. without back-translation, on library-facilities test rows only

This is the number that actually answers whether augmentation helped —
the same 16-20 held-out library-facilities test rows, evaluated once with
the original model and once with the back-translation-augmented model.


In [ ]:
lib_Xte_bt = vectorizer_bt.transform(lib_test['clean'])

comparison_rows = []
for name, model_before, model_after in [
    ('Naive Bayes', nb, nb_bt),
    ('Decision Tree', dt, dt_bt),
    ('Hybrid', hybrid, hybrid_bt),
]:
    pred_before = model_before.predict(lib_Xte)
    pred_after  = model_after.predict(lib_Xte_bt)
    comparison_rows.append({
        'Model': name,
        'Accuracy (before)': accuracy_score(lib_ytrue, pred_before),
        'Accuracy (after back-translation)': accuracy_score(lib_ytrue, pred_after),
        'Macro F1 (before)': f1_score(lib_ytrue, pred_before, average='macro', zero_division=0),
        'Macro F1 (after back-translation)': f1_score(lib_ytrue, pred_after, average='macro', zero_division=0),
    })

bt_comparison = pd.DataFrame(comparison_rows).set_index('Model').round(4)
bt_comparison


Read this table before deciding whether to keep the augmented model.
An improvement here is a genuine, reportable finding — cite the row
count you augmented and the before/after numbers directly. If the
numbers barely move or get worse, that is also worth reporting: it
would mean back-translation mainly added rephrasings of the same
signal the model already had, which is a fair and expected outcome
given what augmentation can and cannot do (see the note at the top of
this section).
